In [0]:
# =============================================================================
# CELL 1: IMPORTS & LOAD BASE TABLES
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, when

# --- Base Tables ---
df_snapshot = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
df_transactions = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
df_ticket_type = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_ticket_type")

In [0]:
# =============================================================================
# CELL 2: FILTER TO AUG 2021+ & AGGREGATE REVENUE/TICKETS BY CC STATUS
# =============================================================================
# A customer can have transactions BOTH as CC and non-CC (they may join/leave).
# We use F.when conditions to split revenue into CC vs non-CC buckets per CDE_ID.
# TransactionDateid format: YYYYMMDD integer
# =============================================================================

df_txn = df_transactions.filter(col("TransactionDateid") >= 20210801)

df_revenue = df_txn.groupBy("CDE_ID").agg(
    # --- Ticket Revenue ---
    F.sum(when((col("TicketFLAG") == "Yes") & (col("CineClubFLAG") == "Yes"), col("Net_Revenue")).otherwise(0)).alias("CC_Ticket_Revenue"),
    F.sum(when((col("TicketFLAG") == "Yes") & (col("CineClubFLAG") == "No"), col("Net_Revenue")).otherwise(0)).alias("NonCC_Ticket_Revenue"),
    # --- Concession Revenue ---
    F.sum(when((col("ConcessionFLAG") == "Yes") & (col("CineClubFLAG") == "Yes"), col("Net_Revenue")).otherwise(0)).alias("CC_Concession_Revenue"),
    F.sum(when((col("ConcessionFLAG") == "Yes") & (col("CineClubFLAG") == "No"), col("Net_Revenue")).otherwise(0)).alias("NonCC_Concession_Revenue"),
    # --- Ticket Counts ---
    F.sum(when((col("TicketFLAG") == "Yes") & (col("CineClubFLAG") == "Yes"), col("Quantity")).otherwise(0)).alias("CC_Tickets"),
    F.sum(when((col("TicketFLAG") == "Yes") & (col("CineClubFLAG") == "No"), col("Quantity")).otherwise(0)).alias("NonCC_Tickets")
)

display(df_revenue)

In [0]:
# =============================================================================
# CELL 3: CLASSIFY CUSTOMERS INTO 3 COHORTS
# =============================================================================
# Cohort 1: "Current Active CC Member" — in latest snapshot with renewal_status LIKE 'Active%'
# Cohort 2: "Former CC Member"         — had CC transactions but NOT currently active
# Cohort 3: "Never CC Member"          — never had a single CC transaction
# =============================================================================

# Step 1: Get currently active CC members from latest snapshot
latest_snapshot = df_snapshot.agg(F.max("snapshot_date_id")).collect()[0][0]
print(f"Latest snapshot date: {latest_snapshot}")

df_active_cc = (
    df_snapshot
    .filter(col("snapshot_date_id") == latest_snapshot)
    .filter(col("renewal_status").like("Active%"))
    .select("cde_id")
    .distinct()
)

# Step 2: Get all CDE_IDs that EVER had a CineClub transaction
df_ever_cc = (
    df_txn
    .filter(col("CineClubFLAG") == "Yes")
    .select(col("CDE_ID").cast("string").alias("cde_id"))
    .distinct()
)

# Step 3: Join to revenue table and assign cohort labels
# (Rename keys before join to avoid column ambiguity)
df_active_cc_renamed = df_active_cc.select(col("cde_id").cast("int").alias("active_cde_id"))
df_ever_cc_renamed = df_ever_cc.select(col("cde_id").cast("int").alias("ever_cc_cde_id"))

df_revenue_status = (
    df_revenue
    .join(df_active_cc_renamed, col("CDE_ID") == col("active_cde_id"), "left")
    .join(df_ever_cc_renamed, col("CDE_ID") == col("ever_cc_cde_id"), "left")
    .withColumn(
        "Current_Status",
        when(col("active_cde_id").isNotNull(), lit("Current Active CC Member"))
        .when(col("ever_cc_cde_id").isNotNull(), lit("Former CC Member"))
        .otherwise(lit("Never CC Member"))
    )
    .drop("active_cde_id", "ever_cc_cde_id")
)

# Verify cohort counts
df_revenue_status.groupBy("Current_Status").count().show()
display(df_revenue_status)

In [0]:
# =============================================================================
# CELL 4: CURRENT ACTIVE CC MEMBER SAVINGS ("YOU HAVE SAVED")
# =============================================================================
# Logic (from CineClub Savings notebook):
#   - Ticket savings   = (discount type 5 & 6 qty) * (GA price - $9.99)
#   - Concession savings = 25% of discounted concession revenue (type 2)
#   - Gaming savings   = 25% of discounted gaming revenue (type 3)
#   - Online booking   = $1 per online ticket visit with CC discount
#   - Adult tickets    = report_code1 = 'AA', childflag = 'No', seniorflag = 'No'
# =============================================================================

%pip install openpyxl -q
import pandas as pd

# --- Step 1: Load GA Prices by Location & Year (from Excel) ---
xlsx_path = "/Volumes/data_exploration_hub/marketing_hub/offline_data/General Admit Prices.xlsx"
df_ga_xlsx = pd.read_excel(xlsx_path, sheet_name="Sheet3")

year_cols = [c for c in df_ga_xlsx.columns if c != "LocationID"]
df_ga_long = (
    df_ga_xlsx
    .melt(id_vars="LocationID", value_vars=year_cols, var_name="visit_year", value_name="ga_ticket_price")
    .rename(columns={"LocationID": "LocationId"})
)
df_ga_long["visit_year"] = df_ga_long["visit_year"].astype(int)
df_ga_prices = spark.createDataFrame(df_ga_long)

# --- Step 2: Build visit-level transactions for active CC members ---
df_cc_visits = (
    df_transactions
    .filter(col("TransactionDateid") >= 20210801)
    .filter(col("CineClubFLAG") == "Yes")
    # Only active CC members
    .join(df_active_cc, df_transactions["CDE_ID"] == df_active_cc["cde_id"], "inner")
    .drop(df_active_cc["cde_id"])
    # Join ticket type for adult ticket filter (report_code1)
    .join(df_ticket_type.select("tickettypeskey", "report_code1", "childflag", "seniorflag"),
          df_transactions["TicketTypeSkey"] == df_ticket_type["tickettypeskey"], "left")
    # Join GA prices by location and year
    .withColumn("visit_year", F.year(F.to_date(col("VisitDateId").cast("string"), "yyyyMMdd")))
    .join(df_ga_prices, on=["LocationId", "visit_year"], how="left")
)

# --- Step 3: Aggregate savings per CDE_ID ---
df_cc_savings = (
    df_cc_visits
    .groupBy("CDE_ID")
    .agg(
        # Ticket savings: CC ticket price is $9.99, saving = (GA price - 9.99) per ticket
        F.round(F.sum(
            when(col("CineClub_Discount_Type_ID").isin(5, 6),
                 col("Quantity") * (F.coalesce(col("ga_ticket_price"), lit(0)) - lit(9.99))
            ).otherwise(0)
        ), 2).alias("ticket_savings"),

        # Concession savings: 25% off concessions (discount type 2)
        F.round(F.sum(
            when((col("ConcessionFLAG") == "Yes") & (col("CineClub_Discount_Type_ID") == 2),
                 col("Net_Revenue") * 0.25
            ).otherwise(0)
        ), 2).alias("concession_savings"),

        # Gaming savings: 25% off gaming/LBE (discount type 3)
        F.round(F.sum(
            when((col("ConcessionFLAG") == "Yes") & (col("CineClub_Discount_Type_ID") == 3),
                 col("Net_Revenue") * 0.25
            ).otherwise(0)
        ), 2).alias("gaming_savings"),

        # Online booking: $1 per online ticket booking with CC discount
        F.sum(
            when((col("TicketFLAG") == "Yes") & (col("COTFLAG") == "Yes") &
                 (col("CineClub_Discount_Type_ID").isin(5, 6)), lit(1)
            ).otherwise(0)
        ).alias("online_booking_savings"),

        # Adult ticket count (report_code1 = AA, not child, not senior)
        F.sum(
            when((col("TicketFLAG") == "Yes") & (col("report_code1") == "AA") &
                 (col("childflag") == "No") & (col("seniorflag") == "No"),
                 col("Quantity")
            ).otherwise(0)
        ).cast("int").alias("CC_Adult_Tickets"),

        # Total ticket count (all types)
        F.sum(when(col("TicketFLAG") == "Yes", col("Quantity")).otherwise(0)
        ).cast("int").alias("CC_Total_Tickets")
    )
    # Total savings = sum of all components
    .withColumn("total_savings", F.round(
        col("ticket_savings") + col("concession_savings") + col("gaming_savings") + col("online_booking_savings"), 2
    ))
)

print(f"Active CC Members with savings: {df_cc_savings.count():,}")
display(df_cc_savings)

In [0]:
# =============================================================================
# CELL 5: NON-CC PROJECTED SAVINGS ("YOU COULD HAVE SAVED")
# =============================================================================
# Target: Everyone EXCEPT current active CC members (Former CC + Never CC)
# Only looks at their non-CC transactions (CineClubFLAG = 'No')
#
# Savings logic (simplified vs actual CC savings):
#   - Ticket savings    = actual revenue paid - (adult ticket count * $9.99), floored at 0
#   - Concession savings = 20% of concession revenue
#   - Adult tickets     = report_code1 = 'AA', childflag = 'No', seniorflag = 'No'
# =============================================================================

# --- Step 1: Get non-CC transactions, exclude active CC members ---
df_noncc_visits = (
    df_transactions
    .filter(col("TransactionDateid") >= 20210801)
    .filter(col("CineClubFLAG") == "No")
    .join(df_active_cc, df_transactions["CDE_ID"] == df_active_cc["cde_id"], "left_anti")
    # Join ticket type for adult filter
    .join(df_ticket_type.select("tickettypeskey", "report_code1", "childflag", "seniorflag"),
          df_transactions["TicketTypeSkey"] == df_ticket_type["tickettypeskey"], "left")
    # Tag as Former vs Never CC
    .join(df_ever_cc.withColumnRenamed("cde_id", "_ever_cde_id"),
          col("CDE_ID") == col("_ever_cde_id"), "left")
    .withColumn("Current_Status",
                when(col("_ever_cde_id").isNotNull(), lit("Former CC Member"))
                .otherwise(lit("Never CC Member")))
    .drop("_ever_cde_id")
)

# --- Step 2: Aggregate per CDE_ID ---
df_noncc_savings = (
    df_noncc_visits
    .groupBy("CDE_ID", "Current_Status")
    .agg(
        # What they actually paid for adult tickets
        F.round(F.sum(
            when((col("TicketFLAG") == "Yes") & (col("report_code1") == "AA") &
                 (col("childflag") == "No") & (col("seniorflag") == "No"),
                 col("Net_Revenue")
            ).otherwise(0)
        ), 2).alias("actual_adult_ticket_revenue"),

        # Adult ticket count
        F.sum(
            when((col("TicketFLAG") == "Yes") & (col("report_code1") == "AA") &
                 (col("childflag") == "No") & (col("seniorflag") == "No"),
                 col("Quantity")
            ).otherwise(0)
        ).cast("int").alias("NonCC_Adult_Tickets"),

        # Total concession revenue
        F.round(F.sum(
            when(col("ConcessionFLAG") == "Yes", col("Net_Revenue")).otherwise(0)
        ), 2).alias("concession_revenue"),

        # Total tickets (all types)
        F.sum(when(col("TicketFLAG") == "Yes", col("Quantity")).otherwise(0)
        ).cast("int").alias("NonCC_Total_Tickets")
    )
)

# --- Step 3: Calculate projected savings ---
# Ticket: what they paid minus what CC price would have been ($9.99/ticket), floor at 0
df_noncc_savings = df_noncc_savings.withColumn(
    "ticket_savings",
    F.round(F.greatest(col("actual_adult_ticket_revenue") - (col("NonCC_Adult_Tickets") * lit(9.99)), lit(0)), 2)
)
# Concession: 20% of what they spent
df_noncc_savings = df_noncc_savings.withColumn(
    "concession_savings",
    F.round(F.greatest(col("concession_revenue") * 0.20, lit(0)), 2)
)
# Total
df_noncc_savings = df_noncc_savings.withColumn(
    "total_could_have_saved",
    F.round(col("ticket_savings") + col("concession_savings"), 2)
)

print(f"Non-CC Members with projected savings: {df_noncc_savings.count():,}")
display(df_noncc_savings)

In [0]:
# =============================================================================
# CELL 6: FORMER CC MEMBER ACTUAL SAVINGS (WHILE THEY WERE CC)
# =============================================================================
# Same savings formula as Cell 4 but for members who are no longer active.
# Only looks at their CC transactions (CineClubFLAG = 'Yes').
# =============================================================================

# --- Step 1: Identify former CC members (ever CC minus currently active) ---
df_former_cc = df_ever_cc.join(df_active_cc, on="cde_id", how="left_anti")

# --- Step 2: Get their CC transactions ---
df_former_cc_visits = (
    df_transactions
    .filter(col("TransactionDateid") >= 20210801)
    .filter(col("CineClubFLAG") == "Yes")
    .join(df_former_cc, df_transactions["CDE_ID"] == df_former_cc["cde_id"], "inner")
    .drop(df_former_cc["cde_id"])
    .join(df_ticket_type.select("tickettypeskey", "report_code1", "childflag", "seniorflag"),
          df_transactions["TicketTypeSkey"] == df_ticket_type["tickettypeskey"], "left")
    .withColumn("visit_year", F.year(F.to_date(col("VisitDateId").cast("string"), "yyyyMMdd")))
    .join(df_ga_prices, on=["LocationId", "visit_year"], how="left")
)

# --- Step 3: Aggregate savings (same logic as Cell 4) ---
df_former_cc_savings = (
    df_former_cc_visits
    .groupBy("CDE_ID")
    .agg(
        F.round(F.sum(
            when(col("CineClub_Discount_Type_ID").isin(5, 6),
                 col("Quantity") * (F.coalesce(col("ga_ticket_price"), lit(0)) - lit(9.99))
            ).otherwise(0)
        ), 2).alias("former_cc_ticket_savings"),

        F.round(F.sum(
            when((col("ConcessionFLAG") == "Yes") & (col("CineClub_Discount_Type_ID") == 2),
                 col("Net_Revenue") * 0.25
            ).otherwise(0)
        ), 2).alias("former_cc_concession_savings"),

        F.round(F.sum(
            when((col("ConcessionFLAG") == "Yes") & (col("CineClub_Discount_Type_ID") == 3),
                 col("Net_Revenue") * 0.25
            ).otherwise(0)
        ), 2).alias("former_cc_gaming_savings"),

        F.sum(
            when((col("TicketFLAG") == "Yes") & (col("COTFLAG") == "Yes") &
                 (col("CineClub_Discount_Type_ID").isin(5, 6)), lit(1)
            ).otherwise(0)
        ).alias("former_cc_online_savings"),

        F.sum(
            when((col("TicketFLAG") == "Yes") & (col("report_code1") == "AA") &
                 (col("childflag") == "No") & (col("seniorflag") == "No"),
                 col("Quantity")
            ).otherwise(0)
        ).cast("int").alias("Former_CC_Adult_Tickets"),

        F.sum(when(col("TicketFLAG") == "Yes", col("Quantity")).otherwise(0)
        ).cast("int").alias("Former_CC_Total_Tickets")
    )
    .withColumn("former_cc_total_savings", F.round(
        col("former_cc_ticket_savings") + col("former_cc_concession_savings") +
        col("former_cc_gaming_savings") + col("former_cc_online_savings"), 2
    ))
)

print(f"Former CC Members with savings: {df_former_cc_savings.count():,}")
display(df_former_cc_savings)

In [0]:
# =============================================================================
# CELL 7: COMPILE FINAL OUTPUT TABLE
# =============================================================================
# Combines all 3 cohorts into one unified table.
# Excludes sentinel CDE_IDs (-1 and NULL) and floors all savings at $0.
#
# Output columns:
#   CDE_ID, Current_Status, CC_Adult_Tickets, CC_Total_Tickets,
#   cc_ticket_savings, cc_concession_savings, cc_gaming_savings,
#   cc_online_booking_savings, you_have_saved, you_could_have_saved
# =============================================================================

# Helper: filter out system sentinel CDE_IDs
def exclude_sentinels(df):
    return df.filter((col("CDE_ID").isNotNull()) & (col("CDE_ID") != -1))

# --- Cohort 1: Current Active CC Members ---
df_final_cc = (
    exclude_sentinels(df_cc_savings)
    .withColumn("Current_Status", lit("Current Active CC Member"))
    .withColumn("total_savings", F.greatest(col("total_savings"), lit(0)))
    .select(
        col("CDE_ID"),
        col("Current_Status"),
        col("CC_Adult_Tickets"),
        col("CC_Total_Tickets"),
        col("ticket_savings").alias("cc_ticket_savings"),
        col("concession_savings").alias("cc_concession_savings"),
        col("gaming_savings").alias("cc_gaming_savings"),
        col("online_booking_savings").alias("cc_online_booking_savings"),
        col("total_savings").alias("you_have_saved"),
        lit(None).cast("double").alias("you_could_have_saved")
    )
)

# --- Cohort 2: Former CC Members (actual CC savings + projected non-CC savings) ---
df_former_projected = (
    df_noncc_savings
    .filter(col("Current_Status") == "Former CC Member")
    .select("CDE_ID", "total_could_have_saved")
)

df_final_former = (
    exclude_sentinels(df_former_cc_savings)
    .join(df_former_projected, on="CDE_ID", how="left")
    .withColumn("Current_Status", lit("Former CC Member"))
    .withColumn("former_cc_total_savings", F.greatest(col("former_cc_total_savings"), lit(0)))
    .withColumn("total_could_have_saved", F.greatest(F.coalesce(col("total_could_have_saved"), lit(0)), lit(0)))
    .select(
        col("CDE_ID"),
        col("Current_Status"),
        col("Former_CC_Adult_Tickets").alias("CC_Adult_Tickets"),
        col("Former_CC_Total_Tickets").alias("CC_Total_Tickets"),
        col("former_cc_ticket_savings").alias("cc_ticket_savings"),
        col("former_cc_concession_savings").alias("cc_concession_savings"),
        col("former_cc_gaming_savings").alias("cc_gaming_savings"),
        col("former_cc_online_savings").alias("cc_online_booking_savings"),
        col("former_cc_total_savings").alias("you_have_saved"),
        col("total_could_have_saved").alias("you_could_have_saved")
    )
)

# --- Cohort 3: Never CC Members (projected savings only) ---
df_final_never = (
    exclude_sentinels(df_noncc_savings.filter(col("Current_Status") == "Never CC Member"))
    .withColumn("total_could_have_saved", F.greatest(col("total_could_have_saved"), lit(0)))
    .select(
        col("CDE_ID"),
        col("Current_Status"),
        lit(0).cast("int").alias("CC_Adult_Tickets"),
        lit(0).cast("int").alias("CC_Total_Tickets"),
        lit(0.0).alias("cc_ticket_savings"),
        lit(0.0).alias("cc_concession_savings"),
        lit(0.0).alias("cc_gaming_savings"),
        lit(0.0).alias("cc_online_booking_savings"),
        lit(0.0).alias("you_have_saved"),
        col("total_could_have_saved").alias("you_could_have_saved")
    )
)

# --- Union all 3 cohorts ---
df_final = df_final_cc.unionByName(df_final_former).unionByName(df_final_never)

# --- Print summary ---
print("FINAL OUTPUT TABLE")
print("=" * 60)
df_final.groupBy("Current_Status").agg(
    F.count("*").alias("count"),
    F.round(F.avg("you_have_saved"), 2).alias("avg_you_have_saved"),
    F.round(F.avg("you_could_have_saved"), 2).alias("avg_you_could_have_saved")
).show(truncate=False)

print(f"Total rows: {df_final.count():,}")
display(df_final)

In [0]:
# =============================================================================
# CELL 7b: FILTER NON-CC MEMBERS TO PAST 12 MONTHS VISITORS ONLY
# =============================================================================
# Keep all Current Active CC Members.
# For Former CC and Never CC, only keep those with at least one transaction
# in the past 12 months (since 2025-07-21).
# =============================================================================

# Identify CDE_IDs with any transaction in P12M
p12m_cutoff = 20250721

df_p12m_visitors = (
    df_transactions
    .filter(col("TransactionDateid") >= p12m_cutoff)
    .select("CDE_ID")
    .distinct()
)

# Split: keep CC members as-is, filter non-CC members
df_final_cc_only = df_final.filter(col("Current_Status") == "Current Active CC Member")
df_final_noncc = df_final.filter(col("Current_Status") != "Current Active CC Member")

# Inner join non-CC with P12M visitors
df_final_noncc_p12m = df_final_noncc.join(df_p12m_visitors, on="CDE_ID", how="inner")

# Recombine
df_final = df_final_cc_only.unionByName(df_final_noncc_p12m)

# Summary
print("AFTER P12M FILTER (Non-CC members only)")
print("=" * 60)
df_final.groupBy("Current_Status").agg(
    F.count("*").alias("count"),
    F.round(F.avg("you_have_saved"), 2).alias("avg_you_have_saved"),
    F.round(F.avg("you_could_have_saved"), 2).alias("avg_you_could_have_saved")
).show(truncate=False)

print(f"Total rows after P12M filter: {df_final.count():,}")

In [0]:
# =============================================================================
# CELL 8: EXPORT FINAL TABLE TO VOLUME AS CSV
# =============================================================================

output_path = "/Volumes/data_exploration_hub/marketing_hub/offline_data/cc_lifetime_savings_sendout.csv"

# Write as single CSV file
df_final.toPandas().to_csv(output_path, index=False)

print(f"CSV written to: {output_path}")
print(f"Rows exported: {df_final.count():,}")